# 02 · No working air conditioning in use

**Objective.** Report the air-conditioning status of New York City residents who died of heat stress after
being exposed to heat at home, as published in Table 2 of the NYC Department of Health and Mental Hygiene's
2026 Heat-Related Mortality Report (2016–2025).

Input: `../inputs/dohmh_heat_report_table2_ac_status.tsv`, the report's underlying Datawrapper table.
Outputs: `data/ac_status.json`, `data/ac_status.csv`.

Heat-stress deaths are deaths caused directly by heat, distinct from the report's modeled estimates of
heat-exacerbated mortality. The denominator is the 25 home-exposed decedents whose AC status was known;
exposure at home does not imply death occurred at home.

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from its own directory (run_notebooks.py does)
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)

import csv

from common import INPUTS, SOURCES, write_json

DENOMINATOR = 25  # Source: DOHMH 2026 Heat-Related Mortality Report, Table 2 (home-exposed decedents with known AC status)


## 1. Read the published table and check it

Three categories, with counts and percentages as published. The counts must total the published denominator
and each percentage must equal count ÷ 25, rounded. Nothing is derived beyond this check.

In [2]:
rows = list(csv.DictReader((INPUTS / "dohmh_heat_report_table2_ac_status.tsv").open(encoding="utf-8"), delimiter="\t"))
categories = [{"label": r["AC status"], "count": int(r["Number"]), "percent": int(r["Percent"].rstrip("%"))} for r in rows]

assert sum(c["count"] for c in categories) == DENOMINATOR, categories
for c in categories:
    assert c["percent"] == round(c["count"] / DENOMINATOR * 100), c
working = next(c for c in categories if c["label"] == "AC working and used")

import pandas as pd
pd.DataFrame(categories)


,label,count,percent
0,No AC in home,13,52
1,AC not working or not in use,12,48
2,AC working and used,0,0


## 2. Write the outputs

The note travels with the data so that any reuse carries the scope: "not working or not in use" includes
units that were present but not running, so it cannot be read as "broken".

In [3]:
write_json(DATA / "ac_status.json", {
    "description": "Air-conditioning status among heat-stress decedents who were exposed to heat at home and whose AC status was known, New York City residents, 2016–2025. Heat-stress deaths are deaths caused directly by heat. Exposure at home does not imply death occurred at home.",
    "sources": [SOURCES["heat_report"]],
    "period": "2016–2025",
    "denominator": DENOMINATOR,
    "working_and_used": working["count"],
    "categories": categories,
    "note": "The denominator is the 25 home-exposed heat-stress decedents with known AC status, not all at-home exposures. 'AC not working or not in use' includes units present but not running; it does not mean every unit was broken.",
})
with (DATA / "ac_status.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["label", "count", "percent"])
    writer.writeheader()
    writer.writerows(categories)
print(f"{working['count']} of {DENOMINATOR} had a working air conditioner in use")


wrote 02_no_working_ac/data/ac_status.json
0 of 25 had a working air conditioner in use
